# mhRaaga bhajan: a sung Krishna bhajan with YuE2 (free Colab GPU)

Set the runtime to **GPU** first (Runtime > Change runtime type), then **Runtime > Run all**. It makes four sung versions (Devanagari and romanised Hindi, two seeds each) and downloads them as MP3, so you can pick the best.

- Here the model **composes its own melody** (cot=full), the natural, human-sounding route. The raga-melody route is in `colab_singer.ipynb`.
- **Honest limits:** the free T4 has 16 GB, the model prefers 24 GB, so low-memory mode is used and it may run out of memory. YuE2 does not list Hindi among its trained languages, so words may blur; the romanised version often sings better.
- **Licence:** YuE2 weights are free for personal and non-commercial use; a company needs a licence (third_party/YuE/README.md).


In [ ]:
!nvidia-smi | head -15

In [ ]:
!git clone --depth 1 https://github.com/multimodal-art-projection/YuE.git
%cd YuE
!pip -q install .   # pins its own PyTorch; the first model use downloads the weights from Hugging Face

In [ ]:
import json
REQUESTS = json.loads(r'''[{"id": "bhajan_devanagari_s7", "style": "Hindi, devotional bhajan, Krishna bhakti, warm male lead vocal, harmonium, bansuri flute, tabla, dholak, manjira, kirtan, gentle, 84 BPM, acoustic, no electronic, no EDM", "lyrics": "[Verse]\nहे मेरे मन के मोहन, हे मेरे घनश्याम\nतेरे चरणों में बीते मेरी हर एक शाम\nन चाहूँ मैं धन-दौलत, न चाहूँ कोई नाम\nबस हृदय में बस जाना, मेरे प्यारे घनश्याम\n\n[Verse]\nजब राहों में अँधेरा हो, तुम दीपक बन जाना\nजब मन मेरा विचलित हो, तुम मुझको समझाना\nन मैं कुछ हूँ, न मेरा कुछ, सब तेरा ही धाम\nमेरा जीवन तुझको अर्पित, मेरे राधे-श्याम\n\n[Chorus]\nजय श्री राधे कृष्ण, जय श्री राधे कृष्ण\nमेरे मन के मोहन, मेरे श्याम", "cot": "full", "seed": 7}, {"id": "bhajan_devanagari_s42", "style": "Hindi, devotional bhajan, Krishna bhakti, warm male lead vocal, harmonium, bansuri flute, tabla, dholak, manjira, kirtan, gentle, 84 BPM, acoustic, no electronic, no EDM", "lyrics": "[Verse]\nहे मेरे मन के मोहन, हे मेरे घनश्याम\nतेरे चरणों में बीते मेरी हर एक शाम\nन चाहूँ मैं धन-दौलत, न चाहूँ कोई नाम\nबस हृदय में बस जाना, मेरे प्यारे घनश्याम\n\n[Verse]\nजब राहों में अँधेरा हो, तुम दीपक बन जाना\nजब मन मेरा विचलित हो, तुम मुझको समझाना\nन मैं कुछ हूँ, न मेरा कुछ, सब तेरा ही धाम\nमेरा जीवन तुझको अर्पित, मेरे राधे-श्याम\n\n[Chorus]\nजय श्री राधे कृष्ण, जय श्री राधे कृष्ण\nमेरे मन के मोहन, मेरे श्याम", "cot": "full", "seed": 42}, {"id": "bhajan_roman_s7", "style": "Hindi, devotional bhajan, Krishna bhakti, warm male lead vocal, harmonium, bansuri flute, tabla, dholak, manjira, kirtan, gentle, 84 BPM, acoustic, no electronic, no EDM", "lyrics": "[Verse]\nhe mere man ke mohan, he mere ghanshyam\ntere charno mein beete meri har ek shaam\nna chahun main dhan-daulat, na chahun koi naam\nbas hriday mein bas jaana, mere pyare ghanshyam\n\n[Verse]\njab raahon mein andhera ho, tum deepak ban jaana\njab man mera vichalit ho, tum mujhko samjhaana\nna main kuchh hun, na mera kuchh, sab tera hi dhaam\nmera jeevan tujhko arpit, mere radhe-shyam\n\n[Chorus]\njai shri radhe krishna, jai shri radhe krishna\nmere man ke mohan, mere shyam", "cot": "full", "seed": 7}, {"id": "bhajan_roman_s42", "style": "Hindi, devotional bhajan, Krishna bhakti, warm male lead vocal, harmonium, bansuri flute, tabla, dholak, manjira, kirtan, gentle, 84 BPM, acoustic, no electronic, no EDM", "lyrics": "[Verse]\nhe mere man ke mohan, he mere ghanshyam\ntere charno mein beete meri har ek shaam\nna chahun main dhan-daulat, na chahun koi naam\nbas hriday mein bas jaana, mere pyare ghanshyam\n\n[Verse]\njab raahon mein andhera ho, tum deepak ban jaana\njab man mera vichalit ho, tum mujhko samjhaana\nna main kuchh hun, na mera kuchh, sab tera hi dhaam\nmera jeevan tujhko arpit, mere radhe-shyam\n\n[Chorus]\njai shri radhe krishna, jai shri radhe krishna\nmere man ke mohan, mere shyam", "cot": "full", "seed": 42}]''')
print(len(REQUESTS), 'versions')

In [ ]:
import time, json, pathlib
from yue2 import YuE2Pipeline
pathlib.Path('out').mkdir(exist_ok=True)
start = time.time()
with YuE2Pipeline.from_pretrained('m-a-p/YuE2-3B', device='cuda', low_vram=True) as pipe:
    for r in REQUESTS:
        req = {k: r[k] for k in ('id', 'style', 'lyrics', 'cot', 'seed')}
        try:
            song = pipe(**req)
            song.save_artifacts('out/' + r['id'])
            print(r['id'], 'done, truncated =', song.truncated, round(time.time() - start), 's')
        except Exception as e:
            print(r['id'], 'FAILED:', type(e).__name__, str(e)[:300])

In [ ]:
import glob, subprocess
from google.colab import files
for f in sorted(glob.glob('out/*/audio.*')):
    mp3 = f.rsplit('/', 1)[0] + '.mp3'
    subprocess.run(['ffmpeg', '-y', '-v', 'error', '-i', f, '-b:a', '192k', mp3], check=True)
    files.download(mp3)